In [ ]:
import os

date = "mother_folder"
rnn_folder = f"base_path/{date}_DL"

shap_number = "project_name"
#os.mkdir(f"{rnn_folder}/model_{shap_number}")

In [ ]:
fs = 20  
calc_start = 5
calc_end = 39
experiments = 175

first_ex = 0
#last_ex = pupil1.shape[0] + pupil2.shape[0] + pupil3.shape[0]
last_ex = 174
NumberOfDatas = last_ex - first_ex + 1        # number of experiments
start_stim = 20
stop_stim = 30
start_ave= 10
end_ave= 20
look_frame = 10               # read the previous and next n frames as input

In [3]:
import numpy as np

peak_idxs = np.load(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_peak_frame.npy")

print(peak_idxs.shape)

(175,)


In [ ]:
import numpy as np
import tqdm

all_evo = []
for ex in tqdm.tqdm(range(experiments)):
    if ex < 10:
        img_path = os.path.join("img_path", f"day_ex0{ex}.npy")
    if ex > 9:
        img_path = os.path.join("img_path", f"day_ex{ex}.npy")
    mov = np.load(img_path)

    #evoked_mov = mov[int(fs*14)-1:int(fs*19)+1]

    peak_idx = peak_idxs[ex]
    evoked_mov = mov[peak_idx-look_frame-1:peak_idx+2]
    all_evo.append(evoked_mov)

print(len(all_evo), all_evo[0].shape)

100%|██████████| 175/175 [00:03<00:00, 51.21it/s]

175 (13, 128, 135)


In [5]:
import numpy as np

#raw_pupil  = np.load(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_pre_pupil.npy")
raw_pupil = np.zeros((len(all_evo), 1))
print(raw_pupil.shape)

(175, 1)


In [6]:
success_ex = np.load(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_selected_experiments_success.npy")
print(success_ex)

failure_ex = np.load(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_selected_experiments_failure.npy")
print(failure_ex)

[  2  22  25  35  43  47  49  54  60  73  91 114 117 121 127 129 133 143
 146 150 152 159 162 165 169 171]
[  4  10  11  13  19  24  37  42  46  57  67  69  74  77  80  83  93  94
 101 103 107 118 123 132 135 157]


In [7]:
raw_pupil[success_ex] = 1
print(raw_pupil[:10])

[[0.]
 [0.]
 [1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]
 [0.]]


In [8]:
pupil = raw_pupil

datasets_x = []
datasets_y = []
for ex in tqdm.tqdm(range(experiments)):
    dataset_x = []
    dataset_y = []
    for f in range(pupil.shape[1]):
        dataset_y.append(pupil[ex, f])
        predata = []
        for step in range(look_frame+1):
            img_3ch = np.stack((all_evo[ex][f+step], all_evo[ex][f+1+step], all_evo[ex][f+2+step]), axis=0)
            predata.append(img_3ch)
        dataset_x.append(predata)
    datasets_x.append(dataset_x)
    datasets_y.append(dataset_y)

datasets_x = np.array(datasets_x)
datasets_y = np.array(datasets_y)

print(datasets_x.shape, datasets_y.shape)

100%|██████████| 175/175 [00:00<00:00, 911.08it/s]


(175, 1, 11, 3, 128, 135) (175, 1)


In [9]:
selected_ex = np.load(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_selected_experiments.npy")
print(selected_ex)

[  2  22  25  35  43  47  49  54  60  73  91 114 117 121 127 129 133 143
 146 150 152 159 162 165 169 171   4  10  11  13  19  24  37  42  46  57
  67  69  74  77  80  83  93  94 101 103 107 118 123 132 135 157]


In [10]:
# Data allocation from experiment No.
TRAIN = list(range(16))
VALID = list(range(5))
TEST  = list(range(5))

In [ ]:
import random


numbers_s = success_ex.tolist()

random.seed(123)

train_numbers_s = random.sample(numbers_s, len(TRAIN))

remaining_s = list(set(numbers_s) - set(train_numbers_s))

valid_numbers_s = random.sample(remaining_s, len(VALID))

test_numbers_s = list(set(remaining_s) - set(valid_numbers_s))

print(len(train_numbers_s), len(valid_numbers_s), len(test_numbers_s))  # ⇒ 40, 10, 10

print(train_numbers_s)
print(valid_numbers_s)
print(test_numbers_s)

np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_numbers_success.npy", np.array(train_numbers_s))
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_valid_numbers_success.npy", np.array(valid_numbers_s))
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_test_numbers_success.npy", np.array(test_numbers_s))

16 5 5
[22, 60, 25, 121, 169, 35, 171, 117, 143, 91, 133, 162, 2, 165, 146, 47]
[127, 150, 49, 73, 129]
[43, 114, 54, 152, 159]


In [ ]:
import random

numbers_f = failure_ex.tolist()

random.seed(123)

train_numbers_f = random.sample(numbers_f, len(TRAIN))

remaining_f = list(set(numbers_f) - set(train_numbers_f))

valid_numbers_f = random.sample(remaining_f, len(VALID))

test_numbers_f = list(set(remaining_f) - set(valid_numbers_f))


print(len(train_numbers_f), len(valid_numbers_f), len(test_numbers_f))  # ⇒ 40, 10, 10

print(train_numbers_f)
print(valid_numbers_f)
print(test_numbers_f)

np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_numbers_failure.npy", np.array(train_numbers_f))
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_valid_numbers_failure.npy", np.array(valid_numbers_f))
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_test_numbers_failure.npy", np.array(test_numbers_f))

16 5 5
[10, 46, 11, 77, 135, 13, 157, 74, 94, 67, 93, 123, 4, 132, 101, 24]
[118, 80, 42, 37, 69]
[103, 107, 19, 83, 57]


In [13]:
#dataset_x = datasets_x[selected_ex]
#dataset_y = datasets_y[selected_ex]

pre_trainX_s = datasets_x[train_numbers_s]
trainY_s = datasets_y[train_numbers_s]
pre_trainX_f = datasets_x[train_numbers_f]
trainY_f = datasets_y[train_numbers_f]

pre_validX_s = datasets_x[valid_numbers_s]
validY_s = datasets_y[valid_numbers_s]
pre_validX_f = datasets_x[valid_numbers_f]
validY_f = datasets_y[valid_numbers_f]

pre_testX_s  = datasets_x[test_numbers_s]
testY_s = datasets_y[test_numbers_s]
pre_testX_f  = datasets_x[test_numbers_f]
testY_f = datasets_y[test_numbers_f]

#print(dataset_x.shape, dataset_y.shape)
print(pre_trainX_s.shape, pre_validX_s.shape, pre_testX_s.shape)
print(pre_trainX_f.shape, pre_validX_f.shape, pre_testX_f.shape)

(16, 1, 11, 3, 128, 135) (5, 1, 11, 3, 128, 135) (5, 1, 11, 3, 128, 135)
(16, 1, 11, 3, 128, 135) (5, 1, 11, 3, 128, 135) (5, 1, 11, 3, 128, 135)


In [14]:
#pre_trainX = dataset_x[train_numbers]
##trainY = dataset_y[train_numbers]
#pre_validX = dataset_x[valid_numbers]
#validY = dataset_y[valid_numbers]
#pre_testX  = dataset_x[test_numbers]
#testY = dataset_y[test_numbers]

pre_trainX = np.concatenate((pre_trainX_s, pre_trainX_f), axis=0)
trainY = np.concatenate((trainY_s, trainY_f), axis=0)
pre_validX = np.concatenate((pre_validX_s, pre_validX_f), axis=0)
validY = np.concatenate((validY_s, validY_f), axis=0)
pre_testX = np.concatenate((pre_testX_s, pre_testX_f), axis=0)
testY = np.concatenate((testY_s, testY_f), axis=0)

print(pre_trainX.shape, pre_validX.shape, pre_testX.shape)

(32, 1, 11, 3, 128, 135) (10, 1, 11, 3, 128, 135) (10, 1, 11, 3, 128, 135)


In [15]:
trainX = pre_trainX.transpose(0,1,2,4,5,3)
print(trainX.shape)

validX = pre_validX.transpose(0,1,2,4,5,3)
print(validX.shape)

testX = pre_testX.transpose(0,1,2,4,5,3)
print(testX.shape)

(32, 1, 11, 128, 135, 3)
(10, 1, 11, 128, 135, 3)
(10, 1, 11, 128, 135, 3)


In [16]:
def reshape_data(x_data, y_data):
    x_data = x_data.reshape(x_data.shape[0]*x_data.shape[1], x_data.shape[2], x_data.shape[3], x_data.shape[4], x_data.shape[5])
    y_data = y_data.reshape(y_data.shape[0]*y_data.shape[1])

    return x_data, y_data

In [17]:
trainX, trainY = reshape_data(trainX, trainY)
validX, validY = reshape_data(validX, validY)
testX, testY   = reshape_data(testX, testY)

print(trainX.shape, trainY.shape)
print(validX.shape, validY.shape)
print(testX.shape, testY.shape)

(32, 11, 128, 135, 3) (32,)
(10, 11, 128, 135, 3) (10,)
(10, 11, 128, 135, 3) (10,)


In [18]:
#np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_signal_min.npy", signal_min)
#np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_signal_max.npy", signal_max)
#np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_signal_ave.npy", signal_ave)
#np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_signal_std.npy", signal_std)

In [19]:
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_features.npy", trainX)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_train_targets.npy", trainY)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_valid_features.npy", validX)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_valid_targets.npy", validY)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_test_features.npy", testX)
np.save(f"{rnn_folder}/model_{shap_number}/{date}_{shap_number}_test_targets.npy", testY)